In [1]:
import os 
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

# 오픈AI 키를 불러온다.
load_dotenv()

# 모델 불러온다.
model = init_chat_model(
    model='gpt-4o-mini',
    temperature=0
)

# Structured output
* `ToolStrategy[StructuredResponseT]`: 구조화된 출력을 위한 도구 호출 사용
* `ProviderStrategy[StructuredResponseT]`: 공급자 기본 구조화된 출력을 사용합니다.
* `type[StructuredResponseT]`: 스키마 유형 - 모델 기능에 따라 자동으로 최상의 전략을 선택합니다.
* `None`: 구조화된 출력 없음

```python
def create_agent(
  ...
  response_format: Union[
    ToolStrategy[StructuredResponseT],
    ProviderStrategy[StructuredResponseT],
    type[StructuredResponseT],
  ]
)
```

* 기본적으로는 하나의 구조화된 출력을 출력합니다.

### Provider strategy
* LLM에서 지원하는 구조화된 출력이 있는 경우 사용할 수 있습니다.

In [2]:
# Pydantic의 BaseModel과 Field를 가져옵니다.
from pydantic import BaseModel, Field

# LangChain의 create_agent 함수를 가져옵니다.
from langchain.agents import create_agent


# 연락처 정보를 구조화해서 저장할 데이터 모델을 정의합니다.
class ContactInfo(BaseModel):
    """Contact information for a person"""

    # 사람의 이름 필드를 정의합니다.
    name: str = Field(description="The name of the person")

    # 사람의 이메일 주소 필드를 정의합니다.
    email: str = Field(description="The email address of the person")

    # 사람의 전화번호 필드를 정의합니다.
    phone: str = Field(description="The phone number of the person")


# 구조화된 출력 형식으로 ContactInfo를 사용하는 에이전트를 생성합니다.
agent = create_agent(
    model=model,
    response_format=ContactInfo # ProviderStrategy가 자동으로 선택됩니다.
)

# 사용자 입력에서 연락처 정보를 추출하도록 에이전트를 실행합니다.
result = agent.invoke({
    "messages": [
        {"role": 'user', "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}
    ]
})

# 구조화된 응답 결과를 출력합니다.
print(result['structured_response'])

# 예상 출력 예시입니다.
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

name='John Doe' email='john@example.com' phone='(555) 123-4567'


### Tool calling strategy

```python
class ToolStrategy(Generic[SchemaT]):
  schema: type[SchemaT]
  tool_message_content: str | None
  handle_errors: Union[
    bool,
    str,
    type[Exception],
    tuple[type[Exception], ...],
    Callable[[Exception], str],
  ]
```

In [5]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

# 개별 상품 리뷰의 분석 결과를 저장할 데이터 모델을 정의합니다. 
class ProductReview(BaseModel):
    """Analysis of a product review"""

    # 상품 평점을 저장하며 1점에서 5점 사이의 값만 허용합니다.
    rating: int | None = Field(description="The rating fo the product", ge=1, le=5)

    # 리뷰 감성을 positive 또는 negative 중 하나로 저장합니다.
    sentiment: Literal['positive', 'negative'] = Field(description='The sentiment of the review')

    # 리뷰의 핵심 포인트를 소문자 기준 1~3단어 문자열 리스트로 저장합니다.
    key_points: list[str] = Field(description="The key points of the review. Lowercase, 1-3 words each.")


# 여러 개의 상품 리뷰 분석 결과를 묶어서 저장할 데이터 모델을 정의합니다.
class ProductReviewList(BaseModel):
    """product review list"""

    # 상품 리뷰 분석 결과 목록을 저장합니다.
    product_reviews: list[ProductReview] = Field(description="List of product reviews")


# 구조화된 출력 형식으로 ProductReviewList를 사용하는 에이전트를 생성합니다.
agent = create_agent(
    model=model,
    # tools = tools,
    response_format=ProductReviewList
)

# 두 개의 리뷰 문장을 분석하도록 에이전트를 실행합니다.
result = agent.invoke({
    "messages": [
        {
            "role": 'user',
            'content': "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive', 'Great product: 5 out of 5 stars. Fast shipping'"
        }
    ]
})


# 구조화된 응답 결과를 확인합니다.
result['structured_response']

# 예상 출력 예시입니다.
# ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive'])

ProductReviewList(product_reviews=[ProductReview(rating=5, sentiment='positive', key_points=['great product', 'fast shipping', 'expensive']), ProductReview(rating=5, sentiment='positive', key_points=['great product', 'fast shipping'])])

In [6]:
# 리스트 내의 메시지를 하나씩 출력
for msg in result['messages']:
    msg.pretty_print()

================================ Human Message =================================

Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive', 'Great product: 5 out of 5 stars. Fast shipping'
================================== Ai Message ==================================

{"product_reviews":[{"rating":5,"sentiment":"positive","key_points":["great product","fast shipping","expensive"]},{"rating":5,"sentiment":"positive","key_points":["great product","fast shipping"]}]}


### Custom tool message content
* 직접 툴을 구성해서 구조화 된 출력을 전달 받을 수 있습니다.

In [7]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

# 회의에서 추출할 액션 아이템 정보를 저장할 데이터 모델을 정의합니다.
class MeetingAction(BaseModel):
    """Action items extracted from a meeting transcript."""

    # 완료해야 할 구체적인 작업 내용을 저장합니다. 
    task: str = Field(description="The specific task to be completed")

    # 해당 작업의 담당자를 저장합니다.
    assignee: str = Field(description="Person responsible for the task")

    # 작업의 우선순위를 low, medium, high 중 하나로 저장합니다.
    priority: Literal['low', 'medium', 'high'] = Field(description="Priority level")


# 구조화된 출력과 사용자 정의 도구 메시지를 사용하는 에이전트를 생성합니다.
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Action item captured and added to meeting notes"
    )
)

# 회의 문장에서 액션 아이템을 추출하도록 에이전트를 실행합니다.
res = agent.invoke({
    "messages": [
        {
            'role': 'user',
            'content': 'From our meeting: Sarah needs to update the project timeline as soon as possible'
        }
    ]
})

In [15]:
# 리스트 내의 메시지를 하나씩 출력
# for msg in res["messages"]:
#     msg.pretty_print()

In [9]:
res["structured_response"]

MeetingAction(task='Update the project timeline', assignee='Sarah', priority='high')

### Error handling
* 에러가 발생하는 경우 스스로 에러를 확인하고 수정할 수 있습니다.

In [11]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

# 연락처 정보를 저장할 데이터 모델을 정의합니다.
class ContactInfo(BaseModel):
    # 사람의 이름을 저장합니다.
    name: str = Field(description="Person's name")

    # 이메일 주소를 저장합니다.
    email: str = Field(description="Email address")


# 이벤트 정보를 저장할 데이터 모델을 정의합니다.
class EventDetails(BaseModel):
    # 이벤트 이름을 저장합니다.
    event_name: str = Field(description="Name of the event")

    # 이벤트 날짜를 저장합니다.
    date: str = Field(description="Event date")


# ContactInfo 또는 EventDetails 중 하나의 형식으로 구조화된 출력을 처리하는 에이전트를 생성합니다.
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(Union[ContactInfo, EventDetails])  # 기본값으로 handle_errors=True가 적용됩니다.
)

# 사용자 입력에서 연락처 또는 이벤트 정보를 추출하도록 에이전트를 실행합니다.
res = agent.invoke({
    "messages": [{"role": "user", "content": "Extract info: John Atwaters (john@gmail.com)"}]

    # 이 메시지는 연락처와 이벤트 정보가 함께 섞여 있어 모델 성능에 따라 원하는 형태로 정확히 추출되지 않을 수 있습니다.
    #"messages": [{"role": "user", "content": "Extract info: John Atwaters (john@gmail.com), Going to Amy's party, May 4th"}]
})

In [16]:
# 리스트 내의 메시지를 하나씩 출력
# for msg in res['messages']:
#     msg.pretty_print()

# 🎯 LangChain Structured Output 복습 문제

## 개요
이 문제는 LangChain의 **Structured Output** 기능을 복습하기 위한 실습 문제입니다.
Pydantic BaseModel을 활용한 구조화된 출력과 다양한 전략(Provider Strategy, Tool Strategy)을 연습합니다.

---

## 📚 핵심 개념 정리

| 전략 | 설명 |
|------|------|
| `response_format=Schema` | 자동으로 최적의 전략 선택 (Provider Strategy 우선) |
| `ToolStrategy(Schema)` | 도구 호출 방식으로 구조화된 출력 생성 |
| `Union[Schema1, Schema2]` | 여러 스키마 중 적합한 것을 자동 선택 |

---

## 🧪 문제 1: 기본 Structured Output (Provider Strategy)

### 시나리오
음식 배달 앱에서 **음식 주문 정보**를 추출하는 에이전트를 만들어야 합니다.

### 요구사항
1. `FoodOrder` Pydantic 모델을 정의하세요:
   - `menu_name`: 메뉴 이름 (str)
   - `quantity`: 수량 (int, 1 이상)
   - `special_request`: 특별 요청사항 (str, 선택적)

2. `create_agent`를 사용하여 에이전트를 생성하세요.
3. 다음 입력에서 주문 정보를 추출하세요:
   > "페퍼로니 피자 2판 주문할게요. 치즈는 많이 넣어주세요."


In [13]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class FoodOrder(BaseModel):
    """음식 주문 정보"""
    menu_name: str = Field(description="주문한 메뉴 이름")
    quantity: int = Field(description="주문한 메뉴 수량", ge=1)
    special_request: str | None = Field(
            default=None,
            description="주문 특별 요청 사항 (선택)"
        )


class OrderList(BaseModel):
    """주문 리스트를 관리합니다."""
    order_list: list[FoodOrder] = Field(description="주문 리스트를 저장합니다.")

# Provider Strategy: response_format에 스키마만 전달하면 자동 선택
agent_q1 = create_agent(
    model=model,
    response_format=OrderList  # Auto-selects ProviderStrategy
)

# 실행
result_q1 = agent_q1.invoke({
    "messages": [{"role": "user", "content": "페퍼로니 피자 2판 주문할게요. 치즈는 많이 넣어주세요."},
                 {"role": "user", "content": "페퍼로니 피자 3판 주문할게요. 페퍼로니 많이 넣어주세요."}]
})

print("=" * 60)
print("문제 1 결과:")
print(result_q1["structured_response"])

문제 1 결과:
order_list=[FoodOrder(menu_name='페퍼로니 피자', quantity=2, special_request='치즈 많이 넣어주세요.'), FoodOrder(menu_name='페퍼로니 피자', quantity=3, special_request='페퍼로니 많이 넣어주세요.')]


## 🧪 문제 2: Tool Strategy와 Literal 타입

### 시나리오
고객 피드백을 분석하여 **감정 분석 결과**를 구조화된 형태로 반환하는 에이전트를 만드세요.

### 요구사항
1. `FeedbackAnalysis` Pydantic 모델을 정의하세요:
   - `category`: 피드백 카테고리 (Literal["product", "service", "delivery", "other"])
   - `sentiment_score`: 감정 점수 (int, 1~10 범위)
   - `keywords`: 핵심 키워드 리스트 (list[str])
   - `requires_followup`: 후속 조치 필요 여부 (bool)

2. `ToolStrategy`를 사용하여 에이전트를 생성하세요.
3. 다음 피드백을 분석하세요:
   > "배송이 너무 늦었어요. 3일이나 걸렸는데 포장도 엉망이었습니다. 다시는 이용 안 할 것 같아요."


In [14]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class FeedbackAnalysis(BaseModel):
    """사용자 피드백 정보를 저장하는 데이터 클래스입니다."""
    category: Literal["product", "service", "delivery", "other"] = Field(description="피드백 카테고리를 저장해주세요.")
    sentiment_score: int = Field(description="감정 점수 (1: 매우 부정적, 10: 매우 긍정적)", ge=1, le=10)
    keywords: list[str] = Field(description="리뷰의 키워드들을 추출해서 저장해주세요.")
    requires_followup: bool = Field(description="후속 조치가 필요한지 여부를 저장해주세요.")

# ToolStrategy 명시적 사용
agent_q2 = create_agent(
    model=model,
    response_format=ToolStrategy(FeedbackAnalysis)
)

# 실행
result_q2 = agent_q2.invoke({
    "messages": [{"role": "user", "content": "배송이 너무 늦었어요. 3일이나 걸렸는데 포장도 엉망이었습니다. 다시는 이용 안 할 것 같아요."}]
})

print("\n" + "=" * 60)
print("문제 2 결과:")
print(result_q2["structured_response"])


문제 2 결과:
category='delivery' sentiment_score=2 keywords=['배송', '늦다', '포장', '엉망'] requires_followup=True
